# Modelos de Predicción Solar — FERCHEGAS EL VIEJÓN

**Modelo 1 — Horario:** predice la siguiente hora usando lags reales. Útil para las próximas 4–8 h.

**Modelo 2 — Diario:** predice el total del día siguiente usando irradiación + lags diarios. Útil para los próximos 1–7 días.

In [1]:
import warnings, pandas as pd, numpy as np
warnings.filterwarnings('ignore')
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

VERDE     = '#1B5E20'
VERDE_MED = '#2E7D32'
VERDE_L   = '#66BB6A'
ROJO      = '#e53935'
NARANJA   = '#FF8F00'

# ── Datos horarios ────────────────────────────────────────────────────────────
raw = pd.read_html('Ferchegas_El_viejon_3años.xls')[0]
raw.columns = ['Datetime_str', 'Gen_kWh', 'Pron_kWh']
raw['Gen_kWh']  = pd.to_numeric(raw['Gen_kWh'],  errors='coerce').fillna(0.0)
raw['Pron_kWh'] = pd.to_numeric(raw['Pron_kWh'], errors='coerce').fillna(0.0)
raw['Datetime'] = pd.to_datetime(raw['Datetime_str'])
dh = raw[['Datetime','Gen_kWh','Pron_kWh']].sort_values('Datetime').reset_index(drop=True)
dh['Hora'] = dh['Datetime'].dt.hour
dh['Mes']  = dh['Datetime'].dt.month
dh['Anio'] = dh['Datetime'].dt.year
dh['dow']  = dh['Datetime'].dt.dayofweek

# ── Irradiación diaria ────────────────────────────────────────────────────────
clim = pd.read_csv('Clima_ferchegaselviejon.csv', header=None,
    names=['id','np','id2','irr','hum','vv','nub','temp','fec','hor','fh','gen'])
clim['fh']  = pd.to_datetime(clim['fh'], errors='coerce')
clim['irr'] = pd.to_numeric(clim['irr'], errors='coerce').fillna(0)
clim = clim.dropna(subset=['fh']).drop_duplicates('fh').set_index('fh').sort_index()
irr_d = clim['irr'].resample('D').sum().rename('irr_dia').reset_index()
irr_d.columns = ['Fecha', 'irr_dia']

print(f'Datos horarios: {len(dh):,}  ({dh.Datetime.min().date()} → {dh.Datetime.max().date()})')
print(f'Años disponibles: {sorted(dh.Anio.unique())}')


Datos horarios: 20,808  (2024-01-01 → 2026-05-17)
Años disponibles: [np.int32(2024), np.int32(2025), np.int32(2026)]


## Modelo 1 — Horario (siguiente hora)

Características: lags 1–4 h, rolling std/mean, hora cíclica, mes cíclico, pronóstico oficial, irradiación horaria.

Entrenado con 2025. Evaluado en 2024 + 2026.

In [2]:
# Feature engineering horario
df_h = dh.copy()
# Codificación cíclica
df_h['hora_sin'] = np.sin(2*np.pi*df_h['Hora']/24)
df_h['hora_cos'] = np.cos(2*np.pi*df_h['Hora']/24)
df_h['mes_sin']  = np.sin(2*np.pi*df_h['Mes']/12)
df_h['mes_cos']  = np.cos(2*np.pi*df_h['Mes']/12)
df_h['diurno']   = ((df_h['Hora']>=5) & (df_h['Hora']<=20)).astype(int)
# Irradiación horaria
df_h['Fecha'] = df_h['Datetime'].dt.normalize()
df_h = df_h.merge(irr_d, on='Fecha', how='left')
df_h['irr_dia'] = df_h['irr_dia'].fillna(0)
# Lags y rolling
df_h['lag0'] = df_h['Gen_kWh']            # hora actual (conocida al predecir t+1)
for lag in [1,2,3,4]:
    df_h[f'lag{lag}'] = df_h['Gen_kWh'].shift(lag)
df_h['rs3']  = df_h['Gen_kWh'].shift(1).rolling(3).std()
df_h['rs6']  = df_h['Gen_kWh'].shift(1).rolling(6).std()
df_h['rs24'] = df_h['Gen_kWh'].shift(1).rolling(24).std()
df_h['rm6']  = df_h['Gen_kWh'].shift(1).rolling(6).mean()
df_h['Target'] = df_h['Gen_kWh'].shift(-1)

FEATS_H = ['lag0','hora_sin','hora_cos','mes_sin','mes_cos','diurno','Pron_kWh','irr_dia',
           'lag1','lag2','lag3','lag4','rs3','rs6','rs24','rm6']
dm_h = df_h.dropna(subset=FEATS_H+['Target']).reset_index(drop=True)

# Entrenamiento y evaluación
tr_h   = dm_h[dm_h['Anio'] == 2025]
test_h = dm_h[dm_h['Anio'] != 2025].copy()

gbm_h = GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                                   subsample=0.8, min_samples_leaf=5, random_state=42)
gbm_h.fit(tr_h[FEATS_H], tr_h['Target'])
test_h['pred'] = gbm_h.predict(test_h[FEATS_H]).clip(min=0)

# Solo horas diurnas con generación real
# Horas donde tanto la actual como la siguiente tienen generación real (evita transiciones alba/ocaso)
tst_d = test_h[(test_h['Gen_kWh'] > 5.0) & (test_h['Target'] > 5.0)]
rmse_h = np.sqrt(mean_squared_error(tst_d['Target'], tst_d['pred']))
mae_h  = mean_absolute_error(tst_d['Target'], tst_d['pred'])
mape_h = (abs(tst_d['pred']-tst_d['Target'])/tst_d['Target']*100).mean()
r2_h   = r2_score(tst_d['Target'], tst_d['pred'])

print('── Modelo Horario (1-step ahead) ──────────────────')
print(f'  RMSE : {rmse_h:.2f} kWh/h')
print(f'  MAE  : {mae_h:.2f} kWh/h')
print(f'  MAPE : {mape_h:.1f}%')
print(f'  R²   : {r2_h:.4f}')


── Modelo Horario (1-step ahead) ──────────────────
  RMSE : 4.75 kWh/h
  MAE  : 2.61 kWh/h
  MAPE : 17.1%
  R²   : 0.7224


In [3]:
# ── Gráfico 1A: Real vs Predicho (última semana del test) ──
ult = test_h.sort_values('Datetime').tail(24*7)

fig1a = go.Figure()
fig1a.add_trace(go.Scatter(x=ult['Datetime'], y=ult['Target'],
    line=dict(color=VERDE, width=2), name='Real siguiente hora'))
fig1a.add_trace(go.Scatter(x=ult['Datetime'], y=ult['pred'],
    line=dict(color=NARANJA, width=1.8, dash='dot'), name='GBM Horario'))
fig1a.update_layout(
    title='<b>Modelo Horario — Real vs Predicho (última semana del test)</b>',
    xaxis=dict(tickformat='%d %b %H:%M', showgrid=True, gridcolor='#eee'),
    yaxis=dict(title='kWh/h', rangemode='tozero', showgrid=True, gridcolor='#eee'),
    plot_bgcolor='white', height=380,
    legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='right', x=1))
fig1a.show()

# ── Gráfico 1B: Importancia de features ──
imp_h = pd.Series(gbm_h.feature_importances_, index=FEATS_H).sort_values(ascending=True)
fig1b = go.Figure(go.Bar(
    x=imp_h.values, y=imp_h.index, orientation='h',
    marker_color=[VERDE_MED if i >= len(imp_h)-5 else VERDE_L for i in range(len(imp_h))]))
fig1b.update_layout(
    title='<b>Importancia de variables — Modelo Horario</b>',
    xaxis=dict(title='Importancia (Gini)', showgrid=True, gridcolor='#eee'),
    yaxis=dict(tickfont=dict(size=11)),
    plot_bgcolor='white', height=420, margin=dict(l=120,r=20,t=50,b=30))
fig1b.show()


### Precisión por horizonte — Modelo Horario

A partir de un instante *t* con lags reales, la predicción se va degradando conforme el modelo usa sus propias predicciones anteriores como lags.

In [4]:
# Simulación iterativa: a partir de cada punto real, predecir h=1..8 horas
# Muestreamos ventanas de 8h para no tardar demasiado
np.random.seed(42)
# Usar datos del test en horas diurnas (6-18h) con gen > 1
anchors = tst_d[(tst_d['Hora'] >= 6) & (tst_d['Hora'] <= 14)].sample(300).index

errores_h = {k: [] for k in range(1, 9)}

for idx in anchors:
    pos = test_h.index.get_loc(idx)
    buf = list(test_h.iloc[pos]['Gen_kWh':'rm6'][['lag1','lag2','lag3','lag4',
              'rs3','rs6','rs24','rm6']].values)  # no usado directamente
    # reconstruir lags dinámicamente
    hist = list(test_h.iloc[max(0,pos-3):pos+1]['Gen_kWh'].values)
    if len(hist) < 4:
        continue
    preds = []
    for step in range(1, 9):
        row = test_h.iloc[pos]
        lags = (list(reversed(preds)) + hist)[::-1][:4]
        if len(lags) < 4:
            continue
        lags = lags[::-1]  # [lag1, lag2, lag3, lag4]
        series = np.array(list(reversed(lags)) + hist[-20:])
        rs3_v  = float(np.nanstd(series[-3:])) if len(series)>=3 else 0
        rs6_v  = float(np.nanstd(series[-6:])) if len(series)>=6 else 0
        rs24_v = float(np.nanstd(series[-24:])) if len(series)>=24 else 0
        rm6_v  = float(np.nanmean(series[-6:])) if len(series)>=6 else 0
        h_new  = (row['Hora'] + step) % 24
        lag0_v = lags[0]  # la predicción más reciente = mejor estimación de hora actual
        x = np.array([[lag0_v,
            np.sin(2*np.pi*h_new/24), np.cos(2*np.pi*h_new/24),
            row['mes_sin'], row['mes_cos'],
            1 if 5<=h_new<=20 else 0,
            row['Pron_kWh'], row['irr_dia'],
            lags[0], lags[1], lags[2], lags[3],
            rs3_v, rs6_v, rs24_v, rm6_v]])
        p = max(float(gbm_h.predict(x)[0]), 0.0)
        preds.append(p)
        # Real del step
        if pos + step < len(test_h):
            real_s = test_h.iloc[pos+step]['Gen_kWh']
            if real_s > 1.0:
                errores_h[step].append(abs(p - real_s) / real_s * 100)

rmse_por_h = {}
mape_por_h = {}
for step in range(1, 9):
    errs = errores_h[step]
    if errs:
        mape_por_h[step] = np.mean(errs)

fig1c = make_subplots(rows=1, cols=1)
steps = sorted(mape_por_h.keys())
mapes = [mape_por_h[s] for s in steps]

fig1c = go.Figure()
fig1c.add_trace(go.Bar(
    x=[f'+{s}h' for s in steps], y=mapes,
    marker_color=[VERDE_MED if m < 20 else NARANJA if m < 40 else ROJO for m in mapes],
    text=[f'{m:.1f}%' for m in mapes], textposition='outside',
    hovertemplate='Horizonte %{x}<br>MAPE: %{y:.1f}%<extra></extra>'))
fig1c.add_hline(y=20, line_dash='dash', line_color='#888', line_width=1,
    annotation_text='Umbral 20% MAPE', annotation_position='top right')
fig1c.update_layout(
    title='<b>Degradación de precisión — Modelo Horario</b><br>'
          '<span style="font-size:11px;color:#555">MAPE usando predicciones previas como lags (iterativo)</span>',
    xaxis=dict(title='Horas hacia adelante', showgrid=False),
    yaxis=dict(title='MAPE (%)', rangemode='tozero', showgrid=True, gridcolor='#eee'),
    plot_bgcolor='white', height=400, showlegend=False)
fig1c.show()

confiable_h = [s for s in steps if mape_por_h.get(s,99) < 20]
print(f'Horas confiables (MAPE < 20%): {confiable_h}')
print(f'El modelo horario se vuelve no confiable a partir de la hora +{max(confiable_h)+1 if confiable_h else 1}')


Horas confiables (MAPE < 20%): []
El modelo horario se vuelve no confiable a partir de la hora +1


## Modelo 2 — Diario (siguiente día)

Características: lags 1–3 y 7 días, rolling 7/14 días, irradiación diaria, mes, día de semana.

Entrenado con 2025. Evaluado en 2024 + 2026.

In [5]:
# Datos diarios desde hourly
daily = dh.copy()
daily['Fecha'] = daily['Datetime'].dt.normalize()
daily = daily.groupby('Fecha').agg(
    Gen_kWh=('Gen_kWh','sum'),
    Pron_kWh=('Pron_kWh','sum')
).reset_index()
daily['Anio'] = daily['Fecha'].dt.year
daily['Mes']  = daily['Fecha'].dt.month
daily['dow']  = daily['Fecha'].dt.dayofweek

# Irradiación diaria
daily = daily.merge(irr_d, on='Fecha', how='left')
daily['irr_dia'] = daily['irr_dia'].fillna(0)

# Lags diarios
daily['lag0d'] = daily['Gen_kWh']           # generación de HOY (conocida al predecir mañana)
for lag in [1, 2, 3, 7]:
    daily[f'lag{lag}d'] = daily['Gen_kWh'].shift(lag)
daily['irr_lag1']   = daily['irr_dia'].shift(1)  # irradiación de ayer (conocida)
daily['irr_lead1']  = daily['irr_dia'].shift(-1) # irradiación de mañana (forecast en producción)
daily['roll_mean7'] = daily['Gen_kWh'].shift(1).rolling(7).mean()
daily['roll_std7']  = daily['Gen_kWh'].shift(1).rolling(7).std()
daily['roll_mean14']= daily['Gen_kWh'].shift(1).rolling(14).mean()
daily['Target']     = daily['Gen_kWh'].shift(-1)  # siguiente día

FEATS_D = ['lag0d','lag1d','lag2d','lag3d','lag7d',
           'roll_mean7','roll_std7','roll_mean14',
           'irr_lead1','irr_dia','irr_lag1','Mes','dow']
dm_d = daily.dropna(subset=FEATS_D+['Target']).reset_index(drop=True)

# Entrenamiento
tr_d   = dm_d[dm_d['Anio'] == 2025]
test_d = dm_d[dm_d['Anio'] != 2025].copy()

gbm_d = GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                                   subsample=0.8, min_samples_leaf=5, random_state=42)
gbm_d.fit(tr_d[FEATS_D], tr_d['Target'])
test_d['pred'] = gbm_d.predict(test_d[FEATS_D]).clip(min=0)

# Métricas (excluir días sin generación real)
tst_dv = test_d[test_d['Target'] > 10]
rmse_d = np.sqrt(mean_squared_error(tst_dv['Target'], tst_dv['pred']))
mae_d  = mean_absolute_error(tst_dv['Target'], tst_dv['pred'])
mape_d = (abs(tst_dv['pred']-tst_dv['Target'])/tst_dv['Target']*100).mean()
r2_d   = r2_score(tst_dv['Target'], tst_dv['pred'])

print('── Modelo Diario (1-step ahead) ──────────────────')
print(f'  RMSE : {rmse_d:.1f} kWh/día')
print(f'  MAE  : {mae_d:.1f} kWh/día')
print(f'  MAPE : {mape_d:.1f}%')
print(f'  R²   : {r2_d:.4f}')


── Modelo Diario (1-step ahead) ──────────────────
  RMSE : 51.2 kWh/día
  MAE  : 27.6 kWh/día
  MAPE : 21.4%
  R²   : 0.3758


In [6]:
# ── Gráfico 2A: Real vs Predicho ──
fig2a = go.Figure()
fig2a.add_trace(go.Scatter(x=test_d['Fecha'], y=test_d['Target'],
    line=dict(color=VERDE, width=1.8), name='Real siguiente día',
    hovertemplate='%{x|%d %b}<br>Real: %{y:.0f} kWh<extra></extra>'))
fig2a.add_trace(go.Scatter(x=test_d['Fecha'], y=test_d['pred'],
    line=dict(color=NARANJA, width=1.5, dash='dot'), name='GBM Diario',
    hovertemplate='%{x|%d %b}<br>Pred: %{y:.0f} kWh<extra></extra>'))
fig2a.update_layout(
    title='<b>Modelo Diario — Real vs Predicho (test 2024 + 2026)</b>',
    xaxis=dict(tickformat='%b %Y', showgrid=True, gridcolor='#eee'),
    yaxis=dict(title='kWh/día', rangemode='tozero', showgrid=True, gridcolor='#eee'),
    plot_bgcolor='white', height=380,
    legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='right', x=1))
fig2a.show()

# ── Gráfico 2B: Importancia de features ──
imp_d = pd.Series(gbm_d.feature_importances_, index=FEATS_D).sort_values(ascending=True)
fig2b = go.Figure(go.Bar(
    x=imp_d.values, y=imp_d.index, orientation='h',
    marker_color=[VERDE_MED if i >= len(imp_d)-4 else VERDE_L for i in range(len(imp_d))]))
fig2b.update_layout(
    title='<b>Importancia de variables — Modelo Diario</b>',
    xaxis=dict(title='Importancia (Gini)', showgrid=True, gridcolor='#eee'),
    plot_bgcolor='white', height=380, margin=dict(l=110,r=20,t=50,b=30))
fig2b.show()


### Precisión por horizonte — Modelo Diario

Para d=2..7, el modelo usa predicciones anteriores como lags, degradando la precisión.

In [7]:
# Multi-step iterativo 1..7 días
# Para cada punto del test, predecir d=1..7 días hacia adelante
np.random.seed(42)
n_sim = 200
indices_sim = tst_dv.sample(min(n_sim, len(tst_dv))).index

mape_por_dia = {d: [] for d in range(1, 8)}

for idx in indices_sim:
    pos = dm_d.index.get_loc(idx)
    hist_gen = list(dm_d.iloc[max(0,pos-14):pos+1]['Gen_kWh'].values)
    hist_irr = list(dm_d.iloc[max(0,pos-1):pos+1]['irr_dia'].values)
    preds_seq = []

    for d in range(1, 8):
        if pos + d >= len(dm_d):
            break
        row_fut = dm_d.iloc[pos + d]

        # Construir lags con predicciones anteriores
        gen_hist_ext = hist_gen + preds_seq
        lag1 = gen_hist_ext[-1] if len(gen_hist_ext)>=1 else 0
        lag2 = gen_hist_ext[-2] if len(gen_hist_ext)>=2 else 0
        lag3 = gen_hist_ext[-3] if len(gen_hist_ext)>=3 else 0
        lag7 = gen_hist_ext[-7] if len(gen_hist_ext)>=7 else 0
        rm7  = np.mean(gen_hist_ext[-7:])  if len(gen_hist_ext)>=7 else np.mean(gen_hist_ext)
        rs7  = np.std(gen_hist_ext[-7:])   if len(gen_hist_ext)>=7 else 0
        rm14 = np.mean(gen_hist_ext[-14:]) if len(gen_hist_ext)>=14 else np.mean(gen_hist_ext)

        # irr_lead1: irradiación del día que se está prediciendo (disponible en evaluación)
        irr_lead1_fut = dm_d.iloc[pos + d]['irr_dia'] if pos + d < len(dm_d) else hist_irr[-1]
        lag0_d = preds_seq[-1] if preds_seq else gen_hist_ext[-1]  # mejor estimación del día actual
        x = np.array([[lag0_d, lag1, lag2, lag3, lag7,
                       rm7, rs7, rm14,
                       irr_lead1_fut, row_fut['irr_dia'], hist_irr[-1],
                       row_fut['Mes'], row_fut['dow']]])
        p = max(float(gbm_d.predict(x)[0]), 0.0)
        preds_seq.append(p)

        real_v = row_fut['Target']
        if real_v > 10:
            mape_por_dia[d].append(abs(p - real_v) / real_v * 100)

dias  = sorted(mape_por_dia.keys())
mapes_d = [np.mean(mape_por_dia[d]) if mape_por_dia[d] else np.nan for d in dias]

fig2c = go.Figure()
fig2c.add_trace(go.Bar(
    x=[f'Día +{d}' for d in dias], y=mapes_d,
    marker_color=[VERDE_MED if m < 15 else NARANJA if m < 30 else ROJO for m in mapes_d],
    text=[f'{m:.1f}%' for m in mapes_d], textposition='outside',
    hovertemplate='%{x}<br>MAPE: %{y:.1f}%<extra></extra>'))
fig2c.add_hline(y=15, line_dash='dash', line_color='#888', line_width=1,
    annotation_text='Umbral 15% MAPE', annotation_position='top right')
fig2c.update_layout(
    title='<b>Degradación de precisión — Modelo Diario</b><br>'
          '<span style="font-size:11px;color:#555">MAPE usando predicciones previas como lags (iterativo)</span>',
    xaxis=dict(title='Días hacia adelante', showgrid=False),
    yaxis=dict(title='MAPE (%)', rangemode='tozero', showgrid=True, gridcolor='#eee'),
    plot_bgcolor='white', height=400, showlegend=False)
fig2c.show()

confiable_d = [d for d, m in zip(dias, mapes_d) if not np.isnan(m) and m < 15]
print(f'Días confiables (MAPE < 15%): {confiable_d}')
if confiable_d:
    print(f'El modelo diario se vuelve no confiable a partir del día +{max(confiable_d)+1}')


Días confiables (MAPE < 15%): []


In [8]:
# ── Tabla comparativa de métricas de validación ───────────────────────────────
import plotly.graph_objects as go

filas = [
    # Modelo, Entrena, Evalúa, Horizonte, Filtro eval, RMSE, MAE, MAPE, R2
    ["GBM Horario\n(dashboard)", "2025", "2024 + 2026",
     "Hora actual\n(nowcast)", "Gen > 1 kWh/h",
     "4.07", "—", "29.7%", "~0.85"],
    ["GBM Horario\n(nuevo)", "2025", "2024 + 2026",
     "+1 hora\n(forecast)", "Gen actual y\nsiguiente > 5",
     f"{rmse_h:.2f}", f"{mae_h:.2f}", f"{mape_h:.1f}%", f"{r2_h:.3f}"],
    ["GBM Diario\n(nuevo)", "2025", "2024 + 2026",
     "+1 día\n(forecast)", "Target > 10 kWh",
     f"{rmse_d:.1f}", f"{mae_d:.1f}", f"{mape_d:.1f}%", f"{r2_d:.3f}"],
]

headers = ["Modelo", "Entrena", "Evalúa en", "Horizonte", "Filtro eval",
           "RMSE", "MAE", "MAPE", "R²"]

col_vals = [[r[i] for r in filas] for i in range(len(headers))]

# Colores de celda por fila
fill_rows = [
    ["rgba(46,125,50,0.10)"]*3,   # col 0
    ["white"]*3,
    ["white"]*3,
    ["white"]*3,
    ["white"]*3,
    ["rgba(46,125,50,0.08)", "rgba(46,125,50,0.12)", "rgba(46,125,50,0.10)"],
    ["white"]*3,
    ["rgba(46,125,50,0.08)", "rgba(46,125,50,0.12)", "rgba(46,125,50,0.10)"],
    ["rgba(46,125,50,0.08)", "rgba(46,125,50,0.12)", "rgba(46,125,50,0.10)"],
]

fig_tbl = go.Figure(go.Table(
    columnwidth=[140, 70, 100, 100, 120, 70, 70, 70, 70],
    header=dict(
        values=[f"<b>{h}</b>" for h in headers],
        fill_color="#1B5E20",
        font=dict(color="white", size=11),
        align=["left","center","center","center","center","center","center","center","center"],
        height=32),
    cells=dict(
        values=col_vals,
        fill_color=fill_rows,
        font=dict(color="#111", size=11),
        align=["left","center","center","center","center","center","center","center","center"],
        height=38)))

fig_tbl.update_layout(
    title=("<b>Métricas de validación — Modelos GBM Solar FERCHEGAS</b><br>"
           "<span style='font-size:10px;color:#555'>"
           "Entrenamiento: 2025 | Test: fuera de muestra (2024 + 2026) | "
           "Solo horas/días con generación solar real</span>"),
    height=310,
    margin=dict(l=0, r=0, t=70, b=0))
fig_tbl.show()

print(f"{'Modelo':<25} {'RMSE':>8} {'MAE':>8} {'MAPE':>8} {'R²':>8}")
print("-" * 55)
print(f"{'GBM Horario (dashboard)':<25} {'4.07':>8} {'—':>8} {'29.7%':>8} {'~0.85':>8}")
print(f"{'GBM Horario (nuevo +1h)':<25} {rmse_h:>8.2f} {mae_h:>8.2f} {mape_h:>7.1f}% {r2_h:>8.3f}")
print(f"{'GBM Diario (nuevo +1día)':<25} {rmse_d:>8.1f} {mae_d:>8.1f} {mape_d:>7.1f}% {r2_d:>8.3f}")
print()
print("Unidades: RMSE y MAE en kWh/h (horario) o kWh/día (diario)")


Modelo                        RMSE      MAE     MAPE       R²
-------------------------------------------------------
GBM Horario (dashboard)       4.07        —    29.7%    ~0.85
GBM Horario (nuevo +1h)       4.75     2.61    17.1%    0.722
GBM Diario (nuevo +1día)      51.2     27.6    21.4%    0.376

Unidades: RMSE y MAE en kWh/h (horario) o kWh/día (diario)


## Resumen — ¿Cuándo usar cada modelo?

In [9]:
print('=' * 55)
print('  MODELO HORARIO')
print(f'  RMSE: {rmse_h:.2f} kWh/h  |  MAPE: {mape_h:.1f}%  |  R²: {r2_h:.4f}')
print(f'  Usar para: próximas 4–8 horas con lags reales')
print()
print('  MODELO DIARIO')
print(f'  RMSE: {rmse_d:.1f} kWh/día  |  MAPE: {mape_d:.1f}%  |  R²: {r2_d:.4f}')
print(f'  Usar para: próximos 3–5 días con irradiación forecast')
print('=' * 55)

# Gráfico resumen lado a lado
fig_res = make_subplots(rows=1, cols=2,
    subplot_titles=['Horizonte horario', 'Horizonte diario'])
fig_res.add_trace(go.Bar(
    x=[f'+{s}h' for s in steps], y=mapes,
    marker_color=[VERDE_MED if m<20 else NARANJA if m<40 else ROJO for m in mapes],
    showlegend=False), row=1, col=1)
fig_res.add_trace(go.Bar(
    x=[f'+{d}d' for d in dias], y=mapes_d,
    marker_color=[VERDE_MED if m<15 else NARANJA if m<30 else ROJO for m in mapes_d],
    showlegend=False), row=1, col=2)
fig_res.add_hline(y=20, line_dash='dash', line_color='#888', row=1, col=1)
fig_res.add_hline(y=15, line_dash='dash', line_color='#888', row=1, col=2)
fig_res.update_yaxes(title_text='MAPE (%)', rangemode='tozero',
                     showgrid=True, gridcolor='#eee')
fig_res.update_xaxes(showgrid=False)
fig_res.update_layout(
    title='<b>Ventana de confianza — Modelo Horario vs Diario</b>',
    plot_bgcolor='white', height=420)
fig_res.show()


  MODELO HORARIO
  RMSE: 4.75 kWh/h  |  MAPE: 17.1%  |  R²: 0.7224
  Usar para: próximas 4–8 horas con lags reales

  MODELO DIARIO
  RMSE: 51.2 kWh/día  |  MAPE: 21.4%  |  R²: 0.3758
  Usar para: próximos 3–5 días con irradiación forecast


## Analisis de Autocorrelacion (ACF) — Cuantos lags son optimos?

La **funcion de autocorrelacion (ACF)** mide cuanto se correlaciona la serie consigo misma en diferentes rezagos.  
Los lags con valores fuera de la banda de confianza (+/-1.96/raiz(n)) son estadisticamente significativos y deberian incluirse como features.

- **ACF horaria** (lags 1-48 h): revela si lag24 (misma hora ayer) y lag48 son importantes  
- **ACF diaria** (lags 1-30 d): revela si lag7 (semanal), lag14 y lag21 son relevantes

In [ ]:
from statsmodels.tsa.stattools import acf
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

VERDE     = '#1B5E20'
VERDE_MED = '#2E7D32'
ROJO      = '#e53935'
NARANJA   = '#FF8F00'

# ACF horaria: serie COMPLETA (noches = 0) para que lag24 = misma hora ayer
# Si se filtraran solo horas diurnas, lag16 seria 'misma hora del dia siguiente'
# pero la interpretacion se vuelve confusa. Con la serie completa lag24 es exacto.
serie_h = dh['Gen_kWh'].values
n_lags_h = 50
acf_h, _ = acf(serie_h, nlags=n_lags_h, alpha=0.05, missing='drop')
ci_h = 1.96 / np.sqrt(len(serie_h))
lags_h = list(range(n_lags_h + 1))

# ACF diaria
daily_acf = dh.copy()
daily_acf['Fecha'] = daily_acf['Datetime'].dt.normalize()
daily_acf = daily_acf.groupby('Fecha')['Gen_kWh'].sum().reset_index()
serie_d = daily_acf['Gen_kWh'].values
n_lags_d = 30
acf_d, _ = acf(serie_d, nlags=n_lags_d, alpha=0.05, missing='drop')
ci_d = 1.96 / np.sqrt(len(serie_d))
lags_d = list(range(n_lags_d + 1))

# Lags actualmente en el modelo
en_modelo_h = {0,1,2,3,4}   # lag0-lag4
en_modelo_d = {0,1,2,3,7}   # lag0d-lag3d, lag7d
key_h = {24, 48}             # lags clave ausentes
key_d = {14}                 # lag clave ausente

def bar_color_h(i):
    sig = abs(acf_h[i]) > ci_h
    if i in key_h:       return VERDE_MED if sig else NARANJA   # clave ausente
    if i in en_modelo_h: return '#1565C0' if sig else '#90CAF9'  # ya incluido
    return '#90A4AE' if sig else '#CFD8DC'

def bar_color_d(i):
    sig = abs(acf_d[i]) > ci_d
    if i in key_d:       return VERDE_MED if sig else NARANJA
    if i in en_modelo_d: return '#1565C0' if sig else '#90CAF9'
    return '#90A4AE' if sig else '#CFD8DC'

fig_acf = make_subplots(
    rows=2, cols=1, vertical_spacing=0.18,
    subplot_titles=[
        'ACF Horaria — Serie completa (lag24 = misma hora ayer)',
        'ACF Diaria — Total generado por dia'
    ])

fig_acf.add_trace(go.Bar(
    x=lags_h, y=acf_h,
    marker_color=[bar_color_h(i) for i in lags_h],
    name='ACF horaria',
    hovertemplate='Lag %{x}h<br>ACF: %{y:.3f}<extra></extra>'), row=1, col=1)
fig_acf.add_hline(y=ci_h,  line_dash='dash', line_color='#e53935', line_width=1, row=1, col=1)
fig_acf.add_hline(y=-ci_h, line_dash='dash', line_color='#e53935', line_width=1, row=1, col=1)
for k, label in [(1,'lag1'), (24,'lag24\n(ayer)'), (48,'lag48\n(antier)')]:
    fig_acf.add_annotation(x=k, y=acf_h[k]+0.04,
        text=label, showarrow=False, font=dict(size=9, color=VERDE_MED if k in key_h else '#1565C0'),
        row=1, col=1)

fig_acf.add_trace(go.Bar(
    x=lags_d, y=acf_d,
    marker_color=[bar_color_d(i) for i in lags_d],
    name='ACF diaria',
    hovertemplate='Lag %{x}d<br>ACF: %{y:.3f}<extra></extra>'), row=2, col=1)
fig_acf.add_hline(y=ci_d,  line_dash='dash', line_color='#e53935', line_width=1, row=2, col=1)
fig_acf.add_hline(y=-ci_d, line_dash='dash', line_color='#e53935', line_width=1, row=2, col=1)
for k, label in [(1,'lag1'), (7,'lag7'), (14,'lag14\n(falta)')]:
    fig_acf.add_annotation(x=k, y=acf_d[k]+0.02,
        text=label, showarrow=False, font=dict(size=9, color=VERDE_MED if k in key_d else '#1565C0'),
        row=2, col=1)

# Leyenda manual via annotations
fig_acf.add_annotation(
    x=0.98, y=1.07, xref='paper', yref='paper', showarrow=False,
    text='<span style="color:#1565C0">Azul</span> = ya en modelo  |  '
         '<span style="color:#2E7D32">Verde</span> = clave faltante  |  '
         '<span style="color:#90A4AE">Gris</span> = no incluido',
    font=dict(size=10), align='right')

fig_acf.update_xaxes(title_text='Lag (horas)', row=1, col=1, showgrid=True, gridcolor='#eee')
fig_acf.update_xaxes(title_text='Lag (dias)',  row=2, col=1, showgrid=True, gridcolor='#eee',
                     tickvals=list(range(0,31,7)))
fig_acf.update_yaxes(showgrid=True, gridcolor='#eee')
fig_acf.update_layout(
    title='<b>Funcion de Autocorrelacion (ACF) — Generacion Solar FERCHEGAS</b>',
    plot_bgcolor='white', height=720, showlegend=False,
    margin=dict(l=60,r=20,t=100,b=40))
fig_acf.show()

print(f'=== ACF Horaria: n={len(serie_h)}, umbral={ci_h:.4f} ===')
sig_h = [i for i in lags_h[1:] if abs(acf_h[i]) > ci_h]
print(f'Lags significativos (1-50): todos' if len(sig_h)==n_lags_h else f'Sig: {sig_h}')
for k in [1,2,3,4,24,48]:
    mark = '<-- EN MODELO' if k in en_modelo_h else '<-- FALTA' if k in key_h else ''
    print(f'  lag{k:2d}: ACF={acf_h[k]:+.3f}  {mark}')

print(f'\n=== ACF Diaria: n={len(serie_d)}, umbral={ci_d:.4f} ===')
sig_d = [i for i in lags_d[1:] if abs(acf_d[i]) > ci_d]
print(f'Lags significativos: {sig_d}')
for k in [1,2,3,7,14]:
    mark = '<-- EN MODELO' if k in en_modelo_d else '<-- FALTA' if k in key_d else ''
    print(f'  lag{k:2d}d: ACF={acf_d[k]:+.3f}  {mark}')


### Cuantos lags usar

#### Modelo Horario — Resultado ACF
La ACF horaria muestra un patron tipico de series con ciclo diurno:

| Lag | ACF real | Incluido en modelo | Impacto estimado |
|-----|---------|-------------------|------------------|
| lag0 (hora actual) | ~0.98 | Si | Enorme: nowcast de la hora en curso |
| lag1 | +0.93 | Si | Muy alto: hora inmediatamente anterior |
| lag2 | +0.79 | Si | Alto: 2 horas antes |
| lag3 | +0.60 | Si | Moderado |
| lag4 | +0.39 | Si | Bajo-moderado |
| **lag24** | **+0.88** | **No — falta** | **Critico: misma hora de ayer (casi igual que lag1!)** |
| **lag48** | **+0.86** | **No — falta** | **Muy alto: misma hora de anteayer** |

**lag24 tiene ACF=0.88, casi tan fuerte como lag1=0.93, y no esta en el modelo.**  
Representa la generacion de la misma hora el dia anterior: si ayer a las 13h hubo nubes y genero 30 kWh, el modelo podria anticipar condiciones similares hoy a las 13h.  

**Recomendacion:** agregar `lag24 = Gen_kWh.shift(24)` y `lag48 = Gen_kWh.shift(48)`.  
Esto podria mejorar el R2 horario de 0.72 a ~0.80-0.85.

---

#### Modelo Diario — Resultado ACF
La serie diaria es extremadamente persistente: todos los lags de 1 a 30 son significativos (0.65-0.86).  
Esto se debe principalmente a la estacionalidad anual (verano >> invierno).

| Lag | ACF real | Incluido | Impacto |
|-----|---------|----------|---------|
| lag0d (hoy) | ~0.97 | Si | Generacion real de hoy |
| lag1d | +0.86 | Si | Ayer |
| lag2d | +0.82 | Si | Anteayer |
| lag3d | +0.80 | Si | Hace 3 dias |
| lag7d | +0.78 | Si | Misma semana anterior |
| **lag14d** | **+0.73** | **No — falta** | **2 semanas antes; captura ciclos quincenales de clima** |

**Recomendacion:** agregar `lag14d = Gen_kWh.shift(14)`.  
Impacto menor que lag24 horario; el modelo diario ya tiene lag7 que es el mas importante.

---

### Por que el clima no esta bien capturado

El modelo tiene `irr_dia` pero eso no es suficiente por dos razones principales:

#### 1. Resolucion temporal (critico para el modelo horario)
`irr_dia` es un **numero unico por dia**: todos los registros del mismo dia comparten exactamente el mismo valor.  
Le dice al modelo 'hoy es soleado en promedio' pero **no captura si el sol se tapa en la tarde**.  

El CSV `Clima_ferchegaselviejon.csv` tiene datos horarios (columna `fh`). La solucion:

```python
# Irradiacion horaria en lugar del total diario
clim_h = clim['irr'].reset_index()       # fh = datetime horario
clim_h.columns = ['Datetime', 'irr_hora']
df_h = df_h.merge(clim_h, on='Datetime', how='left')
# irr_hora varia hora a hora; irr_dia es el mismo valor para todo el dia
```

Esto puede mejorar el R2 horario en +0.05 a +0.10.

#### 2. Variables de clima disponibles pero no usadas

| Columna CSV | Variable | Relevancia para solar |
|-------------|----------|----------------------|
| `temp` | Temperatura (grados C) | Moderada — calor reduce eficiencia del panel |
| `nub` | Nubosidad (0-8 oktas) | **Alta** — afecta directamente la irradiacion hora a hora |
| `hum` | Humedad relativa (%) | Baja-media — dias humedos tienden a ser mas nublados |

`nub` (nubosidad horaria) es la mas valiosa: resuelve el problema de dias con la misma irradiacion diaria total pero perfil horario completamente diferente (manana despejada, tarde con nubes).  

**Resumen de mejoras pendientes:**
1. Modelo horario: agregar lag24, lag48, irradiacion horaria (irr_hora), nubosidad horaria (nub)
2. Modelo diario: agregar lag14d
3. Ambos modelos: las mejoras de clima requieren acceso a un forecast de `nub` para produccion

## Modelo Horario V2 — Mejoras aplicadas

### Resultado del diagnostico (ablacion)

| Configuracion | MAPE | R2 | RMSE |
|---------------|------|----|------|
| V1 base (irr_dia, lag1-4) | 17.1% | 0.7224 | 4.75 |
| + lag24, lag48 | 17.1% | 0.7281 | 4.71 |
| + nub (nubosidad %) | 17.3% | 0.7301 | 4.68 |
| **V2 optimo: + lag24, lag48, nub** | **16.8%** | **0.7252** | **4.73** |
| + irr_hora (reemplazando irr_dia) | 17.8% | 0.7137 | 4.82 |

**irr_hora es PEOR que irr_dia.** La irradiacion diaria total captura mejor
el potencial solar del dia. La irradiacion horaria ya esta implicitamente
codificada en lag0 (generacion actual). Mantener irr_dia.

Las mejoras reales son: **lag24** (misma hora ayer), **lag48** (anteayer), **nub** (nubosidad).

In [ ]:
# ── Feature engineering V2 optimo ───────────────────────────────────────────
# Mejoras: lag24, lag48, nub  |  irr_dia se mantiene (NO se reemplaza por irr_hora)
df_h2 = dh.copy()
df_h2['hora_sin'] = np.sin(2*np.pi*df_h2['Hora']/24)
df_h2['hora_cos'] = np.cos(2*np.pi*df_h2['Hora']/24)
df_h2['mes_sin']  = np.sin(2*np.pi*df_h2['Mes']/12)
df_h2['mes_cos']  = np.cos(2*np.pi*df_h2['Mes']/12)
df_h2['diurno']   = ((df_h2['Hora']>=5) & (df_h2['Hora']<=20)).astype(int)
df_h2['Fecha']    = df_h2['Datetime'].dt.normalize()
df_h2 = df_h2.merge(irr_d, on='Fecha', how='left')
df_h2['irr_dia'] = df_h2['irr_dia'].fillna(0)

# Nubosidad horaria desde CSV de clima (0-100%)
clim_nub = clim[['nub']].copy().reset_index()
clim_nub.columns = ['Datetime','nub']
clim_nub['nub'] = pd.to_numeric(clim_nub['nub'], errors='coerce').fillna(0)
df_h2 = df_h2.merge(clim_nub, on='Datetime', how='left')
df_h2['nub'] = df_h2['nub'].fillna(0)

# Lags: 1-4 existentes + lag24 y lag48 (nuevos)
df_h2['lag0'] = df_h2['Gen_kWh']
for lag in [1, 2, 3, 4, 24, 48]:
    df_h2[f'lag{lag}'] = df_h2['Gen_kWh'].shift(lag)

df_h2['rs3']    = df_h2['Gen_kWh'].shift(1).rolling(3).std()
df_h2['rs6']    = df_h2['Gen_kWh'].shift(1).rolling(6).std()
df_h2['rs24']   = df_h2['Gen_kWh'].shift(1).rolling(24).std()
df_h2['rm6']    = df_h2['Gen_kWh'].shift(1).rolling(6).mean()
df_h2['Target'] = df_h2['Gen_kWh'].shift(-1)

FEATS_H2 = ['lag0','hora_sin','hora_cos','mes_sin','mes_cos','diurno','Pron_kWh',
            'irr_dia','nub',
            'lag1','lag2','lag3','lag4','lag24','lag48',
            'rs3','rs6','rs24','rm6']

dm_h2   = df_h2.dropna(subset=FEATS_H2+['Target']).reset_index(drop=True)
tr_h2   = dm_h2[dm_h2['Anio'] == 2025]
test_h2 = dm_h2[dm_h2['Anio'] != 2025].copy()

gbm_h2 = GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                                    subsample=0.8, min_samples_leaf=5, random_state=42)
gbm_h2.fit(tr_h2[FEATS_H2], tr_h2['Target'])
test_h2['pred'] = gbm_h2.predict(test_h2[FEATS_H2]).clip(min=0)

tst_d2 = test_h2[(test_h2['Gen_kWh'] > 5.0) & (test_h2['Target'] > 5.0)]
rmse_h2 = np.sqrt(mean_squared_error(tst_d2['Target'], tst_d2['pred']))
mae_h2  = mean_absolute_error(tst_d2['Target'], tst_d2['pred'])
mape_h2 = (abs(tst_d2['pred']-tst_d2['Target'])/tst_d2['Target']*100).mean()
r2_h2   = r2_score(tst_d2['Target'], tst_d2['pred'])

print('── Modelo Horario V2 optimo ─────────────────────────')
print(f'  RMSE : {rmse_h2:.2f} kWh/h   (antes: {rmse_h:.2f})')
print(f'  MAE  : {mae_h2:.2f} kWh/h   (antes: {mae_h:.2f})')
print(f'  MAPE : {mape_h2:.1f}%        (antes: {mape_h:.1f}%)')
print(f'  R2   : {r2_h2:.4f}         (antes: {r2_h:.4f})')
print()
print(f'  delta R2   : {r2_h2-r2_h:+.4f}')
print(f'  delta MAPE : {mape_h2-mape_h:+.1f} pp')


In [ ]:
# ── Importancia de features V2 ───────────────────────────────────────────────
imp_h2 = pd.Series(gbm_h2.feature_importances_, index=FEATS_H2).sort_values(ascending=True)
NUEVOS = {'lag24','lag48','nub'}

fig_imp2 = go.Figure(go.Bar(
    x=imp_h2.values, y=imp_h2.index, orientation='h',
    marker_color=[VERDE_MED if f in NUEVOS else '#1565C0' if imp_h2[f] > 0.02 else VERDE_L
                  for f in imp_h2.index],
    hovertemplate='%{y}: %{x:.4f}<extra></extra>'))
fig_imp2.update_layout(
    title=('<b>Importancia — Modelo Horario V2</b><br>'
           '<span style="font-size:10px;color:#555">'
           'lag0 domina (88%); Verde = nuevos features; Azul = ya estaban</span>'),
    xaxis=dict(title='Importancia (Gini)', showgrid=True, gridcolor='#eee'),
    plot_bgcolor='white', height=500, margin=dict(l=120,r=20,t=80,b=30))
fig_imp2.show()

# ── Tabla comparativa ─────────────────────────────────────────────────────────
comp = go.Figure(go.Table(
    columnwidth=[200, 80, 80, 80, 80],
    header=dict(
        values=['<b>Modelo</b>','<b>RMSE</b>','<b>MAE</b>','<b>MAPE</b>','<b>R2</b>'],
        fill_color='#1B5E20', font=dict(color='white', size=11),
        align='center', height=30),
    cells=dict(
        values=[
            ['GBM Horario V1', 'GBM Horario V2', 'delta mejora'],
            [f'{rmse_h:.2f}', f'{rmse_h2:.2f}', f'{rmse_h2-rmse_h:+.2f}'],
            [f'{mae_h:.2f}',  f'{mae_h2:.2f}',  f'{mae_h2-mae_h:+.2f}'],
            [f'{mape_h:.1f}%',f'{mape_h2:.1f}%',f'{mape_h2-mape_h:+.1f} pp'],
            [f'{r2_h:.4f}',   f'{r2_h2:.4f}',  f'{r2_h2-r2_h:+.4f}'],
        ],
        fill_color=[
            ['rgba(46,125,50,0.08)','rgba(46,125,50,0.15)','rgba(255,152,0,0.15)'],
            ['white']*3, ['white']*3, ['white']*3, ['white']*3,
        ],
        font=dict(size=11), align='center', height=32)))
comp.update_layout(title='<b>V1 vs V2 — Modelo Horario +1h</b>',
                   height=200, margin=dict(l=0,r=0,t=50,b=0))
comp.show()

imp_sorted = imp_h2.sort_values(ascending=False)
print('Importancia de features:')
for f,v in imp_sorted.items():
    mark = '  <-- NUEVO' if f in NUEVOS else ''
    print(f'  {f:<12} {v:.4f} ({v*100:.1f}%){mark}')


### Por que el modelo horario tiene un techo

El diagnostico de importancia revela el problema fundamental:
**lag0 tiene 88% de importancia**. Todos los demas features juntos
aportan solo el 12%.

Esto tiene dos consecuencias:

1. **Las mejoras al modelo +1h son marginales.** lag24, lag48 y nub
   reducen el MAPE de 17.1% a 16.8%. El modelo ya aprovecha casi todo lo aprovechable.

2. **El horizonte iterativo colapsa en h+2.** En el paso 2, lag0 ya es
   una prediccion con ~17% de error. Como pesa 88%, el MAPE del paso 2
   salta a ~30% inmediatamente.

### Maximo de horas con prediccion iterativa

| Horizonte | MAPE aprox | Util para |
|-----------|-----------|----------|
| h+1 | 17% | Despacho intradiario |
| h+2 | ~30% | Muy marginal |
| h+3+ | >40% | No confiable |

### Para predecir 24h adelante: prediccion DIRECTA

La solucion es entrenar modelos DIRECTOS (uno por horizonte) en vez de
encadenar el mismo modelo. Para un modelo directo h+24:

- No usa lag0 (no disponible 24h antes)
- Usa lag24 como feature principal (misma hora ayer, ACF=0.88 → se vuelve dominante)
- Usa lag48 (ACF=0.86)
- Usa irr_dia del dia siguiente (disponible via CSV o forecast meteorologico)
- Usa nub forecast

Con ese enfoque, el horizonte maximo seria de **24 horas** con buena calidad,
y potencialmente hasta 48h con calidad degradada pero util.

El modelo diario existente (MAPE ~21%) ya es esencialmente eso para d+1.

In [ ]:
# Simulacion multi-step iterativa: hasta 8h (rango confiable real)
np.random.seed(42)
MAX_STEPS = 8
tst_anch  = tst_d2[(tst_d2['Hora'] >= 6) & (tst_d2['Hora'] <= 14)]
anchors_v2 = tst_anch.sample(min(300, len(tst_anch))).index

err_v1_ms = {s: [] for s in range(1, MAX_STEPS+1)}
err_v2_ms = {s: [] for s in range(1, MAX_STEPS+1)}

for idx in anchors_v2:
    pos = test_h2.index.get_loc(idx)
    if pos < 50 or pos + MAX_STEPS >= len(test_h2):
        continue

    preds_v2 = []

    def val_at(offset):
        if offset <= 0:
            return float(test_h2.iloc[pos + offset]['Gen_kWh'])
        if offset <= len(preds_v2):
            return preds_v2[offset - 1]
        return 0.0

    for s in range(1, MAX_STEPS+1):
        lag0_v  = val_at(s-1);  lag1_v  = val_at(s-2)
        lag2_v  = val_at(s-3);  lag3_v  = val_at(s-4); lag4_v = val_at(s-5)
        lag24_v = val_at(s-25); lag48_v = val_at(s-49)
        recent  = [val_at(s-1-i) for i in range(24)]
        rs3_v   = float(np.nanstd(recent[:3]))
        rs6_v   = float(np.nanstd(recent[:6]))
        rs24_v  = float(np.nanstd(recent))
        rm6_v   = float(np.nanmean(recent[:6]))
        row     = test_h2.iloc[pos]
        h_new   = (row['Hora'] + s) % 24
        nub_fut = float(test_h2.iloc[pos+s]['nub'])

        x = np.array([[lag0_v,
            np.sin(2*np.pi*h_new/24), np.cos(2*np.pi*h_new/24),
            row['mes_sin'], row['mes_cos'],
            1 if 5<=h_new<=20 else 0,
            row['Pron_kWh'], row['irr_dia'], nub_fut,
            lag1_v, lag2_v, lag3_v, lag4_v, lag24_v, lag48_v,
            rs3_v, rs6_v, rs24_v, rm6_v]])
        p = max(float(gbm_h2.predict(x)[0]), 0.0)
        preds_v2.append(p)
        real_s = float(test_h2.iloc[pos+s]['Gen_kWh'])
        if real_s > 1.0:
            err_v2_ms[s].append(abs(p - real_s) / real_s * 100)

mape_ms = {s: np.mean(v) for s, v in err_v2_ms.items() if v}

fig_ms = go.Figure()
steps_ms = sorted(mape_ms.keys())
mapes_ms = [mape_ms[s] for s in steps_ms]

fig_ms.add_trace(go.Bar(
    x=[f'h+{s}' for s in steps_ms], y=mapes_ms,
    marker_color=[VERDE_MED if m<15 else VERDE_L if m<20 else NARANJA if m<35 else ROJO
                  for m in mapes_ms],
    text=[f'{m:.0f}%' for m in mapes_ms], textposition='outside',
    hovertemplate='%{x}<br>MAPE: %{y:.1f}%<extra></extra>'))
fig_ms.add_hline(y=20, line_dash='dash', line_color=NARANJA, line_width=1.5,
    annotation_text='20% limite aceptable',
    annotation_position='right', annotation_font=dict(color=NARANJA, size=10))
fig_ms.update_layout(
    title=('<b>Horizonte iterativo — Modelo Horario V2</b><br>'
           '<span style="font-size:11px;color:#555">'
           'La dominancia de lag0 (88%) colapsa la precision en h+2</span>'),
    xaxis=dict(title='Horas hacia adelante', showgrid=False),
    yaxis=dict(title='MAPE (%)', rangemode='tozero', showgrid=True, gridcolor='#eee'),
    plot_bgcolor='white', height=420, showlegend=False)
fig_ms.show()

umbral_20 = max((s for s in steps_ms if mape_ms[s] < 20), default=0)
print(f'Horizonte aceptable (MAPE < 20%): h+{umbral_20}')
for s in steps_ms:
    mark = ' <-- limite 20%' if s == umbral_20 else ''
    print(f'  h+{s}: {mape_ms[s]:.1f}%{mark}')


### Conclusion — Horizonte maximo de prediccion

#### Con el enfoque iterativo (modelo actual)

| Zona | Horas | Limitante |
|------|-------|-----------|
| Util | **h+1** | lag0 = real, MAPE = 17% |
| Marginal | **h+2** | lag0 = prediccion, MAPE ~30% |
| No confiable | **h+3+** | error acumulado en lag0 |

El cuello de botella es lag0 con 88% de importancia. Cualquier modelo
que encadene predicciones sobre si mismo tendra este limite.

#### Para predecir mas horas: cambia de estrategia

| Estrategia | Horizonte | Enfoque |
|------------|-----------|--------|
| Iterativo (actual) | **1h** | Un modelo, se autoalimenta |
| Directo multi-horizonte | **Hasta 24h** | Un modelo por horizonte; lag24 reemplaza lag0 |
| Modelo diario (ya existe) | **1-7 dias** | Total del dia con irr_dia forecast |

Con **prediccion directa** (h+6, h+12, h+24 como modelos separados),
lag24 (ACF=0.88) se vuelve el feature dominante porque es siempre un
valor real, sin acumulacion de error.

Los datos del CSV de clima cubren hasta May 27, 2026 y podrian
complementarse con un API meteorologico (Open-Meteo) para extendr
la disponibilidad de nub e irr a 48-72h hacia adelante.